# IST8310 校准后飞行包复算

来源：`bags/hardware_20260924_163013_022677`。用户确认磁力计 IST8310、已核对真北、FC 已补偿磁偏角，ROS 额外修正为 0。
该现场确认与 bag 观测分开记录；bag 没有保存磁偏角数值或真北测量参考。

依赖 `mcap`、`mcap-ros2-support`、`numpy`、`pyyaml`。在仓库内启动，所有单元仅读本地文件，不回放 ROS、不连接串口。
分析脚本与输出保存在本目录；完整解释见 README.md。GPS accuracy 是接收机报告值，不是真值误差。


In [1]:
from pathlib import Path
import importlib.util
import json

root = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'agi_ros2/config/hardware.yaml').is_file())
folder = root / 'agi_ros2/analysis/hardware_flight_20260924_163013'
bag = root / 'bags/hardware_20260924_163013_022677'
previous = root / 'agi_ros2/analysis/hardware_diagnostic_20260924/summary.json'

def load_module(name, filename):
    spec = importlib.util.spec_from_file_location(name, filename)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

sensor = load_module('flight_sensor_analysis', folder / 'analyze.py')
msp = load_module('flight_msp_analysis', folder / 'msp_analysis.py')
summary = sensor.summarize(bag)
timeline = msp.summarize(bag, previous)
assert summary == json.loads((folder / 'summary.json').read_text())
assert timeline == json.loads((folder / 'msp_timeline.json').read_text())
print(summary['validation'])
print('MSP evidence:', timeline['validation'])


{'reader': 'mcap.NonSeekingReader with embedded ROS 2 schemas', 'validate_crcs': True, 'crc_scope': 'stored chunk/data CRCs, where present', 'metadata_counts_match': True, 'messages': 154311, 'metadata_messages': 154311, 'topic_count': 27, 'duration_seconds': 130.12318609800002}
MSP evidence: {'metadata_counts_match': True, 'validate_crcs': True, 'messages': 154311}


In [2]:
old = json.loads(previous.read_text())
for field, unit in [('horizontal_accuracy', 'm'), ('vertical_accuracy', 'm'),
                    ('velocity_accuracy', 'm/s')]:
    old_value = old['measurements']['gps_observations/' + field]['p95']
    new_value = summary['measurements']['gps_observations/' + field]['p95']
    print(f'{field}: previous p95={old_value:.4f}; current p95={new_value:.4f} {unit}')
print('Heading validity:', summary['counts']['gps_observations/heading_valid'])
print('Altitude reference:', summary['counts']['gps_observations/altitude_reference'])
print('Fusion initialized:', summary['counts']['/fused_state/initialized'])
print('Raw STATUS distributions:', timeline['raw_status']['distributions'])
print('ARM intervals:', timeline['raw_status']['arm_intervals'])


horizontal_accuracy: previous p95=3.0773; current p95=1.2520 m
vertical_accuracy: previous p95=4.3480; current p95=2.1049 m
velocity_accuracy: previous p95=1.5861; current p95=0.5278 m/s
Heading validity: Counter({'true': 883})
Altitude reference: Counter({'unknown': 883})
Fusion initialized: Counter({'false': 45386})
Raw STATUS distributions: {'flags': {'0': 2282}, 'flag_count': {'30': 2282}, 'rx_failsafe': {'False': 2282}, 'sensor_mask': {'47': 2282}, 'pid_profile': {'1': 2282}, 'rate_profile': {'0': 2282}}
ARM intervals: [{'first_armed_bag_s': 5.256395515, 'last_armed_bag_s': 127.07722373800001, 'first_disarmed_bag_s': 127.117224039, 'sampled_transition_duration_steady_s': 85.71202834799999}]


## 区分墙钟跳变与物理停流

同一 MSP 请求的 tx/rx 共享 request_steady_s，事件 steady_s 和融合 published_steady_time 提供独立时间参照。
ARM 只代表飞控已解锁，不等同于精确离地时长。下面抵扣跳变后的观测间隔仍包含 MAVLink 重新同步过程。


In [3]:
assert len(timeline['clock_jumps']) == 1
jump = timeline['clock_jumps'][0]
assert jump['before']['request_steady_s'] == jump['after']['request_steady_s']
assert jump['before']['code'] == jump['after']['code'] == 150
print('Wall-clock step seconds:', jump['inferred_wall_step_s'])
print('Same request response latency ms:', 1000 * jump['after']['latency_s'])
for stream in ['/sensors/imu', 'gps_observations']:
    raw_gap = summary['streams'][stream]['acquisition_gap_seconds']['maximum']
    print(stream, 'wall gap seconds:', raw_gap,
          'after subtracting wall step:', raw_gap-jump['inferred_wall_step_s'])
print('MSP decoded reason transitions:', timeline['state_reason_transitions'])
print('MSP recorded event error counter maximum:', timeline['errors_counter_max'])


Wall-clock step seconds: 36.14877298256151
Same request response latency ms: 9.851592999993386
/sensors/imu wall gap seconds: 36.558256384 after subtracting wall step: 0.40948340143849293
gps_observations wall gap seconds: 36.650179328 after subtracting wall step: 0.5014063454384896
MSP decoded reason transitions: [{'bag_s': 0.232483704, 'reason': 'Missing configuration readback'}, {'bag_s': 0.43899661100000004, 'reason': 'Unsupported FC/API: require BTFL API 1.48'}, {'bag_s': 80.678594997, 'reason': 'Configuration readback stale'}, {'bag_s': 80.697104979, 'reason': 'MSP transport/request failure; restart required'}]
MSP recorded event error counter maximum: 0
